# IT3091 - Member 2 - IT24100533 - Danthanarayana D.M.R


## Member 2 - Weather and location data

In [4]:
!pip -q install openpyxl joblib

import os
import re
import json
import zipfile
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)

COLAB_REPO = Path("/content/IT3091---Machine-Learning-Assignment")
REPO_PATH = COLAB_REPO if COLAB_REPO.exists() else Path.cwd()  # Colab, or local (notebook opened from repo folder)
RAW_PATH = REPO_PATH / "Raw Datasets"
PREPROCESSED_PATH = REPO_PATH / "Preprocessed Datasets"

PREPROCESSED_PATH.mkdir(parents=True, exist_ok=True)

print("Raw data:", RAW_PATH)
print("Preprocessed data:", PREPROCESSED_PATH)

Raw data: /Users/ravindudanthanarayana/Desktop/IT3091---Machine-Learning-Assignment/Raw Datasets
Preprocessed data: /Users/ravindudanthanarayana/Desktop/IT3091---Machine-Learning-Assignment/Preprocessed Datasets


In [5]:
WEATHER_FILE = RAW_PATH / "weatherData.csv"
LOCATION_FILE = RAW_PATH / "locationData.csv"

if not WEATHER_FILE.exists():
    matches = list(RAW_PATH.glob("*weatherData*.csv"))
    if matches:
        WEATHER_FILE = matches[0]

if not LOCATION_FILE.exists():
    matches = list(RAW_PATH.glob("*locationData*.csv"))
    if matches:
        LOCATION_FILE = matches[0]

if not WEATHER_FILE.exists() or not LOCATION_FILE.exists():
    raise FileNotFoundError("weatherData.csv or locationData.csv was not found in Raw Datasets.")

print("Weather file:", WEATHER_FILE.name)
print("Location file:", LOCATION_FILE.name)

Weather file: weatherData.csv
Location file: locationData.csv


In [6]:
weather = pd.read_csv(WEATHER_FILE)
locations = pd.read_csv(LOCATION_FILE)

weather = weather.loc[:, ~weather.columns.astype(str).str.lower().isin(["index", "unnamed: 0"])]
locations = locations.loc[:, ~locations.columns.astype(str).str.lower().isin(["index", "unnamed: 0"])]

weather = weather.merge(
    locations[["location_id", "city_name"]],
    on="location_id",
    how="left",
    validate="many_to_one"
)

if weather["city_name"].isna().any():
    raise ValueError("Some location_id values did not match locationData.csv.")

weather["date"] = pd.to_datetime(weather["date"], errors="coerce")
weather = weather.dropna(subset=["date"]).copy()

print("Weather shape:", weather.shape)
display(weather.head())

Weather shape: (142371, 22)


,location_id,date,weather_code (wmo code),temperature_2m_max (°C),temperature_2m_min (°C),temperature_2m_mean (°C),apparent_temperature_max (°C),apparent_temperature_min (°C),apparent_temperature_mean (°C),daylight_duration (s),sunshine_duration (s),precipitation_sum (mm),rain_sum (mm),precipitation_hours (h),wind_speed_10m_max (km/h),wind_gusts_10m_max (km/h),wind_direction_10m_dominant (°),shortwave_radiation_sum (MJ/m²),et0_fao_evapotranspiration (mm),sunrise,sunset,city_name
0,0,2010-01-01,1,30.1,22.6,26.0,34.5,25.0,29.0,42220.20,38905.73,0.0,0.0,0,12.2,27.4,19,20.92,4.61,06:22,18:05,Colombo
1,0,2010-01-02,51,30.1,23.7,26.3,33.9,26.1,29.7,42225.71,37451.01,0.1,0.1,1,13.0,27.0,24,17.71,3.91,06:22,18:06,Colombo
2,0,2010-01-03,51,29.6,23.1,26.0,34.5,26.2,29.9,42231.68,33176.43,0.6,0.6,3,12.3,27.4,17,17.76,3.66,06:22,18:06,Colombo
3,0,2010-01-04,2,28.9,23.1,25.7,31.7,26.1,28.4,42238.11,38289.20,0.0,0.0,0,17.0,34.6,357,16.50,3.75,06:23,18:07,Colombo
4,0,2010-01-05,1,28.1,21.3,24.6,30.0,22.9,26.2,42244.99,39113.82,0.0,0.0,0,18.7,37.1,353,23.61,5.00,06:23,18:07,Colombo


In [7]:
def season_and_year(dt):
    m, y = dt.month, dt.year

    if m in [4, 5, 6, 7, 8]:
        return pd.Series(["Yala", y])

    if m in [9, 10, 11, 12]:
        return pd.Series(["Maha", y])

    if m in [1, 2, 3]:
        return pd.Series(["Maha", y - 1])

    return pd.Series([pd.NA, pd.NA])

weather[["Season", "Year"]] = weather["date"].apply(season_and_year)
weather["Year"] = pd.to_numeric(weather["Year"], errors="coerce").astype("Int64")
weather = weather[weather["Year"].between(2012, 2023)].copy()

display(weather[["date", "Season", "Year"]].head())

,date,Season,Year
821,2012-04-01,Yala,2012
822,2012-04-02,Yala,2012
823,2012-04-03,Yala,2012
824,2012-04-04,Yala,2012
825,2012-04-05,Yala,2012


In [8]:
WEATHER_COLS = [
    "temperature_2m_mean (°C)",
    "precipitation_sum (mm)",
    "wind_speed_10m_max (km/h)",
    "shortwave_radiation_sum (MJ/m²)",
    "et0_fao_evapotranspiration (mm)"
]

for c in WEATHER_COLS:
    weather[c] = pd.to_numeric(weather[c], errors="coerce")

weather_daily_nat = (
    weather.groupby(["date", "Season", "Year"], as_index=False)[WEATHER_COLS]
           .mean()
)

weather_season = (
    weather_daily_nat.groupby(["Year", "Season"], as_index=False)
    .agg(
        SeasonMeanTemp_C=("temperature_2m_mean (°C)", "mean"),
        SeasonTotalPrecip_mm=("precipitation_sum (mm)", "sum"),
        SeasonMeanWind_kmh=("wind_speed_10m_max (km/h)", "mean"),
        SeasonMeanSolar_MJm2=("shortwave_radiation_sum (MJ/m²)", "mean"),
        SeasonTotalET0_mm=("et0_fao_evapotranspiration (mm)", "sum"),
        WeatherDays=("date", "nunique")
    )
)

display(weather_season.tail(10))

,Year,Season,SeasonMeanTemp_C,SeasonTotalPrecip_mm,SeasonMeanWind_kmh,SeasonMeanSolar_MJm2,SeasonTotalET0_mm,WeatherDays
14,2019,Maha,25.286281,1221.614815,15.525995,18.702238,844.108889,213
15,2019,Yala,27.137085,602.092593,19.153232,19.594607,701.223333,153
16,2020,Maha,25.094392,1391.496296,16.164832,17.787799,796.865926,212
17,2020,Yala,26.812709,690.314815,17.526846,19.334619,661.879259,153
18,2021,Maha,25.023008,1186.703704,15.790391,18.831791,835.437037,212
19,2021,Yala,26.357081,850.007407,18.162890,19.374212,656.260000,153
20,2022,Maha,24.578616,1171.896296,15.951013,18.837386,827.268889,212
21,2022,Yala,26.052554,857.596296,18.655604,19.427855,651.735556,153
22,2023,Maha,25.355208,1769.181481,15.518362,18.635681,839.443704,213
23,2023,Yala,26.854829,588.022222,18.153813,20.775805,717.528519,153


In [9]:
weather_output = PREPROCESSED_PATH / "02_weather_preprocessed.csv"
weather_season.to_csv(weather_output, index=False)

print("Saved:", weather_output)
print("Rows:", len(weather_season))

Saved: /Users/ravindudanthanarayana/Desktop/IT3091---Machine-Learning-Assignment/Preprocessed Datasets/02_weather_preprocessed.csv
Rows: 24
